# GNSS Paper 1 - overnight GPU pipeline (generalization + defense)

Runs `23_generalization.py` and `24_defense.py` directly (cell 4 below), chunked into rounds sized to fit Kaggle's 12h session cap:

- `23_generalization.py` (full, all 14 models incl. RBF-kernel SVM) -> `generalization.csv`
- `24_defense.py` (diagnostic adversarial-training baseline, 6 DL -- classical models have no gradient-based AT analogue, so this is DL-only by design) -> `defense_baseline.csv`

Each round runs ONE protocol/chunk (see cell 4's comment block) so a completed commit always preserves its Output; a timeout kill preserves nothing. Both scripts have their own resume/skip logic keyed off whatever is already in `results/tables/`, so re-running is safe.

## Run it overnight WITHOUT losing the result
Use **Save Version -> Save & Run All (Commit)** (top-right), NOT the interactive Run. Commit mode runs headless in the background and stores everything in `/kaggle/working/` as that version's **Output**, downloadable for good after the session ends. The last cell also PRINTS both CSVs as text as a backup. You can close your laptop.

**Before you commit, set in the right panel:** Accelerator = **GPU**, Internet = **On**, and **Add Input** = your dataset with `texbat_track_combined.csv`.

## 1. Clone the code (Paper-1 branch)

`Master_Thesis_Part_A` is a **private** repo, so an anonymous clone fails with
`could not read Username for 'https://github.com'`. Before running the next
cell, attach your GitHub token as a Kaggle Secret (NOT pasted into a cell --
a notebook cell can end up shared or public, a Secret cannot):

1. This notebook's right-hand panel -> **Add-ons -> Secrets**.
2. **Add a new secret**: label it `GITHUB_TOKEN`, value = your Personal Access
   Token (fine-grained, scoped to just this repo, `Contents: Read-only` is
   enough to clone).
3. Toggle it **Attached** for this notebook, then save.

The next cell reads it via Kaggle's secrets API at runtime; the token itself
never appears in the notebook source.

In [ ]:
import os, subprocess, sys

REPO_HOST = "github.com/Ojerinde/Master_Thesis_Part_A.git"
BRANCH    = "paper1-experiment"
DST       = "/kaggle/working/repo"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = f"https://{token}@{REPO_HOST}"
    print("Using GITHUB_TOKEN from Kaggle Secrets.")
except Exception as e:
    REPO = f"https://{REPO_HOST}"
    print(f"[WARN] No GITHUB_TOKEN secret found ({e}). Trying an anonymous clone, "
          f"which will fail with 'could not read Username' on a private repo. "
          f"See the markdown cell above to attach one.")

if not os.path.exists(DST):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, DST], check=True)
os.chdir(DST)
print("cwd:", os.getcwd()); print("top-level:", sorted(os.listdir("."))[:20])

## 2. Place the corpus CSV where the loader expects it

In [ ]:
import glob, shutil, os
src = glob.glob("/kaggle/input/**/texbat_track_combined.csv", recursive=True)
assert src, "Attach the Kaggle dataset that contains texbat_track_combined.csv (right panel > Add Input)."
os.makedirs("data/processed", exist_ok=True)
shutil.copy(src[0], "data/processed/texbat_track_combined.csv")
print(f"CSV placed: {os.path.getsize('data/processed/texbat_track_combined.csv'):,} bytes")

## 2b. (Only if resuming) seed partial results from a previous killed session

**Kaggle does NOT preserve `/kaggle/working` when a session hits the 12h timeout** -- only a run that COMPLETES successfully keeps its Output. So if a previous attempt was killed by the timeout, there is nothing to resume from automatically.

The safe pattern is to run in **deliberately small pieces that each finish inside 12h**:
1. First commit: `--protocol cross_scenario` (3 folds, a few hours). It completes, so its Output IS preserved -- download `generalization.csv` from that version's Output tab.
2. Add that downloaded `generalization.csv` as a Kaggle **Dataset** (or just re-upload it into the SAME dataset you use for the corpus CSV), attach it as an input to a NEW notebook version, and run this cell to seed it in before running `--protocol leave_prn`.
3. Repeat in further chunks if 11 PRN folds still do not fit one session (`--protocol leave_prn` resumes from whatever is already in the seeded file, skipping completed PRNs).

If this is your FIRST run, skip this cell (there is nothing to seed yet).

In [ ]:
import glob, shutil, os, pandas as pd
# Look for previously downloaded result files in any attached input dataset,
# and seed them into results/tables/ so the resume/skip logic in
# 23_generalization.py / 24_defense.py continues instead of restarting.
os.makedirs("results/tables", exist_ok=True)
for fname in ["generalization.csv", "defense_baseline.csv"]:
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if hits:
        shutil.copy(hits[0], f"results/tables/{fname}")
        g = pd.read_csv(f"results/tables/{fname}")
        key = 'protocol' if fname == 'generalization.csv' else 'model'
        print(f"Seeded {fname} from {hits[0]}  ({len(g)} rows, "
              f"done so far: {sorted(g[key].unique().tolist())})")
    else:
        print(f"No previous {fname} found in attached inputs -- starting fresh.")

## 3. Environment check (do NOT `pip install -r requirements.txt` here)

In [ ]:
import sys, subprocess, importlib, torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY"))
if not torch.cuda.is_available():
    print("[WARN] No GPU detected. Set Accelerator = GPU in the right panel, then re-run.")
for mod, pip_name in [("imblearn","imbalanced-learn"),("xgboost","xgboost"),("lightgbm","lightgbm"),("sklearn","scikit-learn")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=True)
print("deps OK")

## 4. Run the full pipeline (smoke -> full generalization -> full defense)

In [ ]:
import os, subprocess, sys, time

# ================================================================
# THE ONLY CELL YOU EDIT BETWEEN ROUNDS. Kaggle's 12h session cap does NOT
# preserve output on a timeout -- only a run that COMPLETES keeps its Output.
# So each round below is sized to finish comfortably inside 12h; chunk further
# with PRN_LIMIT / MODELS if a round still runs long (check the printed
# per-fold/per-model timing in THIS round's log to size the next one).
#
# ROUND 1: STAGE="generalization", PROTOCOL="cross_scenario"   (3 folds, ~4-6h)
# ROUND 2: STAGE="generalization", PROTOCOL="leave_prn", PRN_LIMIT=5 (chunk of 5 PRNs)
# ROUND 3+: repeat ROUND 2 (resume skips finished PRNs) until all 11 are done
# ROUND N: STAGE="defense", MODELS="CNN-1D,LSTM,BiLSTM"        (chunk of 3 models)
# ================================================================
STAGE      = "generalization"   # "generalization" | "defense"
PROTOCOL   = "cross_scenario"   # generalization only: "cross_scenario" | "leave_prn"
PRN_LIMIT  = None                # generalization/leave_prn only: int or None (=all)
MODELS     = None                # defense only: "CNN-1D,LSTM" or None (=all not done)
BATCH_SIZE = 256                 # both stages: config default 32 under-uses the GPU

# PYTHONUTF8=1: some stages print non-ASCII (checkmarks etc.); with stdout
# redirected to a pipe, Python can fall back to a non-UTF-8 codec and crash
# with UnicodeEncodeError. Force UTF-8 mode regardless of platform/locale.
env = dict(os.environ, PYTHONPATH=".", PYTHONWARNINGS="ignore", PYTHONUTF8="1")
if STAGE == "generalization":
    cmd = [sys.executable, "-u", "experiments/23_generalization.py",
           "--protocol", PROTOCOL, "--batch-size", str(BATCH_SIZE)]
    if PRN_LIMIT is not None:
        cmd += ["--prn-limit", str(PRN_LIMIT)]
else:
    cmd = [sys.executable, "-u", "experiments/24_defense.py",
           "--batch-size", str(BATCH_SIZE)]
    if MODELS:
        cmd += ["--models", MODELS]

print("running:", " ".join(cmd))
t0 = time.time()
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1, encoding="utf-8")
for line in p.stdout:
    print(line, end="")
p.wait()
print(f"[{STAGE} done] exit={p.returncode}  elapsed={(time.time()-t0)/60:.1f} min")
assert p.returncode == 0, f"{STAGE} failed with exit {p.returncode} (scroll up for the traceback)."

## 5. Save + PRINT both result tables (recoverable even if a file is lost)

In [ ]:
import shutil, os, pandas as pd
pd.set_option('display.max_rows', None); pd.set_option('display.width', 200)
for name in ['generalization.csv', 'defense_baseline.csv']:
    srcp = f'results/tables/{name}'
    if not os.path.exists(srcp):
        print(f'[missing] {srcp}'); continue
    dst = f'/kaggle/working/{name}'; shutil.copy(srcp, dst)
    g = pd.read_csv(dst)
    print('='*70); print(f'{name}  rows={len(g)}'); print('='*70)
    print(g.round(4).to_string(index=False))
    print(f'----BEGIN {name}----'); print(g.to_csv(index=False)); print(f'----END {name}----')
print('Both saved to /kaggle/working/ -> download from the committed version Output tab.')

### After it finishes
Download `generalization.csv` and `defense_baseline.csv` from the committed version's **Output** tab (persist indefinitely), or copy the text between the `BEGIN/END` markers. Send both back.
- generalization: must have **both** `classical` and `deep` rows; ds2 stays the collapse case.
- defense: expect adversarial training to lift recall under FGSM/PGD only partially, with the detector still falling at eps=0.2 (partial protection, not a fix).